# MolFormer SMILES Encoder Evaluation

This notebook evaluates the pretrained MolFormer-based SMILES encoder implemented in `src/encoders/smiles_encoder.py`.

The goal is to verify the encoder with the real pretrained MolFormer model and examine its behavior on representative SMILES inputs. The evaluation includes output shape and validity, single-molecule and batch behavior, determinism, embedding similarity, and inference performance.

These checks complement the unit tests in `tests/test_smiles_encoder.py`, which use lightweight mock objects to test the encoder wrapper without loading the pretrained model.

## 1. Setup

Import the libraries needed for the evaluation and add the project's `src` directory to the Python path so the MolFormer `SmilesEncoder` can be imported from the repository.

The evaluation will use PyTorch for model execution and tensor operations, along with timing utilities for later inference-performance measurements.

In [ ]:
import sys
from pathlib import Path
import time

import torch

# Locate the ai project directory from the notebook folder
AI_ROOT = Path.cwd().parent
SRC_DIR = AI_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from encoders.smiles_encoder import SmilesEncoder

print(f"AI project root: {AI_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load the Pretrained MolFormer Encoder

Initialize the `SmilesEncoder` using the pretrained MolFormer model. The encoder is kept frozen for this evaluation because the current implementation uses MolFormer as a pretrained feature extractor rather than fine-tuning its weights.

The model is placed in evaluation mode to ensure consistent inference behavior. Since CUDA is not available in the current environment, this evaluation will run on the CPU.

In [ ]:
encoder = SmilesEncoder(freeze=True)
encoder.eval()

device = next(encoder.model.parameters()).device

print(f"Model: {encoder.model_name}")
print(f"Device: {device}")
print(f"Output dimension: {encoder.out_dim}")
print(f"Maximum sequence length: {encoder.max_length}")
print(
    f"Trainable parameters: "
    f"{sum(p.numel() for p in encoder.model.parameters() if p.requires_grad):,}"
)

## 3. Single-Molecule Encoding

Test the pretrained MolFormer encoder on a single SMILES string. Ethanol (`CCO`) is used as a simple representative molecule.

The encoder should treat a single string as a batch of size one and return one 768-dimensional molecular representation. The resulting embedding is also checked for finite values to confirm that the real model produces a numerically valid output.

In [ ]:
single_smiles = "CCO"

with torch.no_grad():
    single_embedding = encoder(single_smiles)

print(f"Input SMILES: {single_smiles}")
print(f"Embedding shape: {tuple(single_embedding.shape)}")
print(f"Data type: {single_embedding.dtype}")
print(f"All values finite: {torch.isfinite(single_embedding).all().item()}")

## 4. Batch Encoding

Evaluate MolFormer on a batch of different SMILES strings to verify that the encoder can process multiple molecules simultaneously.

The batch contains several representative molecular structures of different sizes and compositions. The encoder should return one 768-dimensional representation for each input molecule, with all embedding values remaining finite.

In [ ]:
batch_smiles = [
    "CCO",                       # ethanol
    "CC(=O)O",                   # acetic acid
    "c1ccccc1",                  # benzene
    "CC(=O)OC1=CC=CC=C1C(=O)O", # aspirin
]

with torch.no_grad():
    batch_embeddings = encoder(batch_smiles)

print(f"Number of molecules: {len(batch_smiles)}")
print(f"Embedding shape: {tuple(batch_embeddings.shape)}")
print(f"Data type: {batch_embeddings.dtype}")
print(f"All values finite: {torch.isfinite(batch_embeddings).all().item()}")

## 5. Encoding Consistency

Verify that MolFormer produces consistent representations for identical inputs.

First, the same SMILES string is encoded twice to test deterministic behavior in evaluation mode. Second, the molecule is encoded once as a single string and once as a one-item batch to confirm that the encoder's single-input handling does not change the resulting molecular representation.

The maximum absolute difference between each pair of embeddings is reported in addition to a direct numerical comparison.

In [ ]:
with torch.no_grad():
    first_embedding = encoder("CCO")
    second_embedding = encoder("CCO")

    single_input_embedding = encoder("CCO")
    one_item_batch_embedding = encoder(["CCO"])

deterministic = torch.allclose(first_embedding, second_embedding)
single_batch_match = torch.allclose(
    single_input_embedding,
    one_item_batch_embedding,
)

deterministic_max_diff = (
    first_embedding - second_embedding
).abs().max().item()

single_batch_max_diff = (
    single_input_embedding - one_item_batch_embedding
).abs().max().item()

print(f"Repeated encoding matches: {deterministic}")
print(f"Maximum repeated-encoding difference: {deterministic_max_diff:.8f}")
print()
print(f"Single input matches one-item batch: {single_batch_match}")
print(f"Maximum single/batch difference: {single_batch_max_diff:.8f}")

## 6. Molecular Embedding Similarity

Compare MolFormer representations for several different molecules using cosine similarity.

Cosine similarity measures how closely two embedding vectors point in the same direction. A value closer to 1 indicates more similar representations, while a lower value indicates less similar representations.

This analysis checks that MolFormer does not collapse different molecules into identical representations and provides an initial view of how molecular differences are reflected in the pretrained embedding space. It is intended as a qualitative sanity check rather than a direct measure of chemical similarity.

In [ ]:
import torch.nn.functional as F

similarity_smiles = {
    "ethanol": "CCO",
    "propanol": "CCCO",
    "butanol": "CCCCO",
    "benzene": "c1ccccc1",
    "aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
}

names = list(similarity_smiles.keys())
smiles = list(similarity_smiles.values())

with torch.no_grad():
    similarity_embeddings = encoder(smiles)

normalized_embeddings = F.normalize(
    similarity_embeddings,
    p=2,
    dim=1,
)

similarity_matrix = (
    normalized_embeddings @ normalized_embeddings.T
)

print("Cosine similarity matrix:\n")

print(" " * 12 + "".join(f"{name:>12}" for name in names))

for name, row in zip(names, similarity_matrix):
    values = "".join(f"{value.item():>12.4f}" for value in row)
    print(f"{name:<12}{values}")

## 7. CPU Inference Performance

Measure MolFormer inference time for individual and batched SMILES inputs.

The model is warmed up before timing so that initialization overhead does not dominate the measurements. The same set of molecules is then encoded either one at a time or together as a batch. Multiple repetitions are used to obtain a more stable average.

Because CUDA is not available in the current environment, these measurements represent CPU performance only. They provide a baseline for evaluating batching efficiency and for possible future comparison with alternative SMILES encoders such as a BiLSTM.

In [ ]:
timing_smiles = [
    "CCO",
    "CC(=O)O",
    "c1ccccc1",
    "CCCO",
    "CCCCO",
    "CCN",
    "CCOC",
    "CC(=O)OC1=CC=CC=C1C(=O)O",
]

repetitions = 10

# Warm up the model before measuring inference time
with torch.no_grad():
    _ = encoder(timing_smiles)

# Measure encoding each molecule separately
start = time.perf_counter()

with torch.no_grad():
    for _ in range(repetitions):
        for smiles in timing_smiles:
            _ = encoder(smiles)

individual_seconds = time.perf_counter() - start

# Measure encoding all molecules together
start = time.perf_counter()

with torch.no_grad():
    for _ in range(repetitions):
        _ = encoder(timing_smiles)

batch_seconds = time.perf_counter() - start

total_molecules = len(timing_smiles) * repetitions

individual_ms_per_molecule = (
    individual_seconds / total_molecules * 1000
)

batch_ms_per_molecule = (
    batch_seconds / total_molecules * 1000
)

speedup = individual_seconds / batch_seconds

print(f"Device: {device}")
print(f"Molecules per batch: {len(timing_smiles)}")
print(f"Repetitions: {repetitions}")
print()
print(f"Individual encoding total: {individual_seconds:.3f} s")
print(f"Individual average: {individual_ms_per_molecule:.2f} ms/molecule")
print()
print(f"Batch encoding total: {batch_seconds:.3f} s")
print(f"Batch average: {batch_ms_per_molecule:.2f} ms/molecule")
print()
print(f"Batching speedup: {speedup:.2f}x")

## 8. BACE1 Dataset Evaluation

Load the BACE1 bioactivity dataset using MuFu's shared `pipeline.bioactivity_loader`. This is the same loader used by the ECFP4 and GNN encoder evaluations, ensuring that MolFormer is evaluated on the same standardized dataset.

The loader retrieves BACE1 IC50 measurements for ChEMBL target `CHEMBL4822`, applies the project's filtering and molecule-level aggregation procedure, and caches the processed dataset for reuse. The resulting data contains standardized SMILES, pIC50 regression targets, and binary activity labels using the project's default pIC50 threshold of 6.0.

Using the shared loader keeps this notebook reproducible and avoids maintaining a separate dataset-processing implementation for MolFormer.

In [ ]:
from pipeline import bioactivity_loader as bl

bace_df = bl.load_bace1()

print(f"BACE1 molecules: {len(bace_df):,}")
print(f"Columns: {list(bace_df.columns)}")
print()
print("pIC50 summary:")
print(bace_df["pIC50"].describe().round(3))
print()
print("Activity labels:")
print(bace_df["active"].value_counts().sort_index())

bace_df.head()

## 9. Encode the BACE1 Dataset

Encode all standardized BACE1 SMILES using the pretrained MolFormer encoder. The molecules are processed in batches to avoid excessive memory usage and to reflect how the encoder can be used on larger datasets in the MuFu pipeline.

The resulting batch embeddings are concatenated into a single tensor containing one 768-dimensional representation for each BACE1 molecule. The total encoding time is measured to provide an initial CPU inference-performance reference.

In [ ]:
batch_size = 32
bace_smiles = bace_df["smiles"].tolist()

embedding_batches = []

start_time = time.perf_counter()

with torch.no_grad():
    for start in range(0, len(bace_smiles), batch_size):
        batch = bace_smiles[start : start + batch_size]
        batch_embeddings = encoder(batch)
        embedding_batches.append(batch_embeddings.cpu())

        processed = min(start + batch_size, len(bace_smiles))
        if processed % 1000 < batch_size or processed == len(bace_smiles):
            print(f"Encoded {processed:,} / {len(bace_smiles):,} molecules")

bace_embeddings = torch.cat(embedding_batches, dim=0)

elapsed_time = time.perf_counter() - start_time

print()
print(f"Embedding shape: {tuple(bace_embeddings.shape)}")
print(f"All values finite: {torch.isfinite(bace_embeddings).all().item()}")
print(f"Encoding time: {elapsed_time:.2f} seconds")
print(f"Average time per molecule: {1000 * elapsed_time / len(bace_smiles):.2f} ms")

## 10. BACE1 Embedding Validation

Validate the embeddings produced for the full BACE1 dataset. In addition to checking their shape and numerical validity, examine variation across molecules to confirm that the encoder produces distinct molecular representations rather than a constant or degenerate output.

The standard deviation across molecules provides a simple sanity check of embedding variation, while cosine similarity between selected molecule pairs provides an additional view of how the learned representations differ.

In [ ]:
embedding_std = bace_embeddings.std(dim=0)

print(f"Mean feature standard deviation: {embedding_std.mean().item():.6f}")
print(f"Minimum feature standard deviation: {embedding_std.min().item():.6f}")
print(f"Maximum feature standard deviation: {embedding_std.max().item():.6f}")
print(f"Features with zero variation: {(embedding_std == 0).sum().item()} / {encoder.out_dim}")

unique_embeddings = torch.unique(bace_embeddings, dim=0)

print(f"Unique embeddings: {len(unique_embeddings):,} / {len(bace_embeddings):,}")

## 11. Duplicate Embedding Check

The full BACE1 evaluation produced 8,086 unique embeddings for 8,087 molecules. This indicates that one pair of dataset entries produced identical MolFormer representations.

Identify the duplicated embeddings and inspect their corresponding BACE1 records to determine whether this is explained by duplicate or equivalent SMILES inputs rather than an encoder failure.

In [ ]:
# Find embeddings that occur more than once
_, inverse_indices, counts = torch.unique(
    bace_embeddings,
    dim=0,
    return_inverse=True,
    return_counts=True,
)

duplicate_groups = torch.where(counts > 1)[0]

for group in duplicate_groups:
    row_indices = torch.where(inverse_indices == group)[0].cpu().tolist()

    print(f"Duplicate embedding at rows: {row_indices}")
    display(
        bace_df.iloc[row_indices][
            ["molecule_chembl_id", "smiles", "pIC50", "active"]
        ]
    )

In [ ]:
smiles_a = bace_df.iloc[1639]["smiles"]
smiles_b = bace_df.iloc[1640]["smiles"]

print(f"SMILES 1639: {smiles_a}")
print(f"SMILES 1640: {smiles_b}")
print(f"SMILES identical: {smiles_a == smiles_b}")
print(f"Length 1639: {len(smiles_a)}")
print(f"Length 1640: {len(smiles_b)}")

In [ ]:
encoded_pair = encoder.tokenizer(
    [smiles_a, smiles_b],
    padding=True,
    truncation=True,
    max_length=encoder.max_length,
    return_tensors="pt",
)

tokens_identical = torch.equal(
    encoded_pair["input_ids"][0],
    encoded_pair["input_ids"][1],
)

print(f"Original SMILES identical: {smiles_a == smiles_b}")
print(f"Original lengths: {len(smiles_a)}, {len(smiles_b)}")
print(f"Tokenized sequence length: {encoded_pair['input_ids'].shape[1]}")
print(f"Tokenized inputs identical after truncation: {tokens_identical}")

## 12. Conclusion

The MolFormer SMILES encoder successfully passed both unit-level and real-model evaluation. The encoder supports single SMILES strings and batches, produces 768-dimensional molecular representations, returns finite outputs, and behaves deterministically in evaluation mode.

Evaluation on the full BACE1 dataset successfully encoded all 8,087 molecules into an embedding tensor with shape `(8087, 768)`. All embedding values were finite, and all 768 embedding features showed nonzero variation across the dataset, indicating that the encoder produces non-degenerate molecular representations.

On the current CPU environment, encoding the full BACE1 dataset took approximately 7.38 minutes, corresponding to an average of 54.74 ms per molecule. These measurements provide a baseline for future comparisons with alternative SMILES encoders on the same hardware.

Of the 8,087 BACE1 entries, 8,086 unique embeddings were produced. Investigation of the single duplicate embedding showed that the two corresponding SMILES strings were different but exceeded the configured maximum sequence length of 202 tokens. After tokenization and truncation, their input token sequences were identical, explaining the identical MolFormer representations. This is expected behavior under the current truncation configuration.

Overall, the evaluation confirms that the MolFormer SMILES encoder is functioning correctly with the pretrained model and the project's BACE1 data and is ready for integration into the MuFu pipeline. The observed long-sequence truncation behavior should be considered in future encoder comparisons or configuration tuning.